# Random Forest：共同输入非线性基准

这是第一个正式非线性模型。它与共同输入 Logistic 使用相同的 126 个公共特征和固定 Train/Selection 切分；数值缺失、缺失指示与类别 One-Hot 全部只在 Train 上学习。候选森林覆盖不同树数、深度、叶节点约束、特征抽样和样本抽样强度。Selection 用于锁模及与共同输入 Logistic 的配对比较，Test 保持封存。

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'configs' / 'experiment.yaml').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.comparison import load_locked_selection_prediction
from credit_risk.development import load_development_data
from credit_risk.models.random_forest import run_random_forest
print('Project root:', PROJECT_ROOT)

Project root: /Users/ganlu/Desktop/home-credit-risk-english


In [2]:
development = load_development_data(PROJECT_ROOT)
assert not hasattr(development, 'X_test')
assert not hasattr(development, 'y_test')
reference_selection_prediction = load_locked_selection_prediction(
    development, 'common_logistic'
)
display(development.split_overview)
print('公共特征数:', len(development.feature_columns))
print('已对齐的共同输入 Logistic Selection 预测数:', len(reference_selection_prediction))

,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test (sealed),46127,NaN


公共特征数: 126
已对齐的共同输入 Logistic Selection 预测数: 46127


In [3]:
result = run_random_forest(
    development,
    reference_selection_prediction=reference_selection_prediction,
    register_lock=True,
)
print('随机森林训练与锁模完成。')

RF candidate 1 [Train-OOB/conservative] OOB AR=0.438891, fit=2.7s


RF candidate 2 [Train-OOB/shallow_feature_rich] OOB AR=0.471353, fit=15.8s


RF candidate 3 [Train-OOB/balanced] OOB AR=0.448833, fit=4.5s


RF candidate 4 [Train-OOB/feature_rich_regularized] OOB AR=0.475820, fit=21.9s


RF candidate 5 [Train-OOB/flexible] OOB AR=0.447525, fit=8.5s


RF candidate 6 [Train-OOB/deep_regularized] OOB AR=0.452898, fit=4.3s


RF candidate 7 [full_train_finalist/feature_rich_regularized] Selection AR=0.490839, gap=0.124014, fit=479.9s


RF candidate 8 [full_train_finalist/shallow_feature_rich] Selection AR=0.479992, gap=0.057587, fit=230.5s


RF candidate 9 [tree_count_refinement/feature_rich_regularized] Selection AR=0.491019, gap=0.124304, fit=742.9s


随机森林训练与锁模完成。


In [4]:
display(result.search_results)
display(result.metrics)
display(result.uplift_vs_logistic)
display(result.raw_feature_importance.head(30))
display(result.selection_deciles)
display(result.locked_config)
print('锁模产物:', result.run_directory)

,candidate,stage,profile,max_depth,min_samples_leaf,min_samples_split,max_features,max_samples,bootstrap,n_estimators,...,train_auc,selection_auc,train_ar,selection_ar,ar_gap,train_ks,selection_ks,selection_ap,selection_logloss,eligible_for_lock
0,9,tree_count_refinement,feature_rich_regularized,12.0,100,200,0.5,0.70,True,500,...,0.807662,0.745509,0.615323,0.491019,0.124304,0.455745,0.363550,0.231328,0.530740,True
1,7,full_train_finalist,feature_rich_regularized,12.0,100,200,0.5,0.70,True,300,...,0.807426,0.745419,0.614852,0.490839,0.124014,0.455457,0.363571,0.231673,0.530813,True
2,8,full_train_finalist,shallow_feature_rich,8.0,50,100,0.5,0.70,True,300,...,0.768789,0.739996,0.537579,0.479992,0.057587,0.395216,0.351753,0.223850,0.571117,False
3,1,train_oob_screen,conservative,8.0,100,200,sqrt,0.70,True,160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,2,train_oob_screen,shallow_feature_rich,8.0,50,100,0.5,0.70,True,160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
5,3,train_oob_screen,balanced,12.0,50,100,sqrt,0.85,True,160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
6,4,train_oob_screen,feature_rich_regularized,12.0,100,200,0.5,0.70,True,160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
7,5,train_oob_screen,flexible,16.0,20,40,sqrt,0.85,True,160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
8,6,train_oob_screen,deep_regularized,NaN,100,200,sqrt,0.70,True,160,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


,sample,n,bad_rate,AUC,AR,KS,AP,Logloss
0,Train,215257,0.080727,0.807426,0.614852,0.455457,0.283088,0.516323
1,Selection,46127,0.080734,0.745419,0.490839,0.363571,0.231673,0.530813


,model,reference_model,metric,difference,lower,upper,bootstrap_se,confidence_level,bootstrap_replicates,higher_is_better
0,random_forest,common_logistic,AUC,-0.002723,-0.006001,0.000431,0.001663,0.95,500,True
1,random_forest,common_logistic,AR,-0.005447,-0.012003,0.000861,0.003325,0.95,500,True
2,random_forest,common_logistic,KS,-0.002907,-0.012071,0.006025,0.004753,0.95,500,True
3,random_forest,common_logistic,AP,-0.003757,-0.009762,0.001830,0.002947,0.95,500,True
4,random_forest,common_logistic,Logloss,0.282138,0.279313,0.284726,0.001382,0.95,500,False


,raw_feature,importance
0,EXT_SOURCE_3,0.245945
1,EXT_SOURCE_2,0.239289
2,EXT_SOURCE_1,0.069993
3,GOODS_CREDIT_RATIO,0.037415
4,EMPLOYED_YEARS,0.034741
5,AGE_YEARS,0.031402
6,CREDIT_ANNUITY_RATIO,0.029343
7,EMPLOYED_AGE_RATIO,0.023735
8,AMT_ANNUITY,0.018573
9,ID_PUBLISH_YEARS,0.016896


,risk_group,rank_min,rank_max,n,bad_n,bad_rate,probability_min,probability_max,lift,cumulative_bad_capture
0,1,1,4613,4613,1197,0.259484,0.656799,0.921546,3.214077,0.321429
1,2,4614,9226,4613,659,0.142857,0.559770,0.656785,1.769487,0.498389
2,3,9227,13839,4613,479,0.103837,0.480089,0.559681,1.286168,0.627014
3,4,13840,18451,4612,364,0.078925,0.416096,0.480079,0.977592,0.724758
4,5,18452,23064,4613,307,0.066551,0.359631,0.416074,0.824329,0.807197
5,6,23065,27677,4613,227,0.049209,0.307221,0.359627,0.609520,0.868153
6,7,27678,32289,4612,200,0.043365,0.259282,0.307205,0.537138,0.921858
7,8,32290,36902,4613,138,0.029915,0.213309,0.259260,0.370545,0.958915
8,9,36903,41515,4613,90,0.019510,0.164845,0.213305,0.241660,0.983083
9,10,41516,46127,4612,63,0.013660,0.044495,0.164788,0.169199,1.000000


{'model_name': 'random_forest',
 'experiment_track': 'controlled',
 'data_sha256': '52e96b895b1112e1c853f670e58372719c8441c5ed1c57ac2f7fad559d784f5f',
 'split_sha256': '4ae4ceee661ba8c210fa0a1cf950448799a0aef468659e6d2590c1bb4e8a3835',
 'feature_policy_version': 1,
 'training_sample': 'Train only',
 'selection_role': 'candidate selection and paired comparison only',
 'test_accessed': False,
 'parameters': {'n_estimators': 300,
  'max_depth': 12,
  'min_samples_leaf': 100,
  'min_samples_split': 200,
  'max_features': 0.5,
  'max_samples': 0.7,
  'bootstrap': True,
  'class_weight': 'balanced_subsample',
  'random_state': 42},
 'selected_structure_profile': 'feature_rich_regularized',
 'candidate_count': 9,
 'screening_sample_size': 60000,
 'screening_finalists': ['feature_rich_regularized', 'shallow_feature_rich'],
 'final_full_train_fit_reused_from_selection_candidate': True,
 'raw_feature_count': 126,
 'numeric_feature_count': 111,
 'categorical_feature_count': 15,
 'encoded_feature_

锁模产物: /Users/ganlu/Desktop/home-credit-risk-english/artifacts/random_forest/20260831T035909_287983Z


In [5]:
registry_path = PROJECT_ROOT / development.configs['experiment']['test_policy']['registry_path']
registry = json.loads(registry_path.read_text(encoding='utf-8'))
assert registry['models']['random_forest']['locked'] is True
assert registry['models']['random_forest']['config']['test_accessed'] is False
assert registry['suite_locked'] is False
print('随机森林已锁定；其余非线性模型尚未全部登记，Test 继续封存。')

随机森林已锁定；其余非线性模型尚未全部登记，Test 继续封存。
